# CWD prion quantification — Phase 1 (classical, no training, no GPU)

Quantifies PrP burden in DAB (brown) IHC photomicrographs of cervidized-mouse brain
(GtDeer / GtElk treatment vs WT control), 4x, ~686 images.

1. **Burden** — % PrP-positive tissue area, object count, density per mm^2
2. **Morphometry (coarse at 4x)** — per-object area, circularity, solidity
3. **Spread** — object centroids for spatial stats

Pipeline: **00** (scale from Olympus SIS metadata) -> **01** (this) -> **02** (StarDist, optional).

**Key notes**
- Fixed `DAB_THRESHOLD` (not per-image Otsu) so burden is comparable across images/groups.
- Calibrated at 0.05: WT control ~0.04% (background), treatment ~6%.
- At ~5 um/px, burden + regional distribution are solid; per-plaque morphotype is coarse.
- The **animal** is the experimental unit; the trailing filename number is an IMAGE index,
  NOT an animal. Stats stay image-level (exploratory) until `_ANIMAL_KEY` is filled.

In [ ]:
# If needed:  pip install numpy pandas matplotlib scikit-image scipy tifffile
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import io, color, filters, morphology, measure, segmentation, util
from skimage.transform import rescale
from skimage.feature import peak_local_max
from IPython.display import display
print('skimage', __import__('skimage').__version__, '| requires >= 0.20')

In [ ]:
# ======================= CONFIG — EDIT THIS CELL =======================
DATA_DIR = Path('/Users/ryaneastman/Desktop/prion_images')   # searched recursively
OUT_DIR  = Path('./outputs'); OUT_DIR.mkdir(exist_ok=True)

# All images are 4x (single magnification) -> no resampling needed.
TARGET_UM_PER_PX = None     # None = keep native scale (no resample); set a value only for MIXED magnification
MIN_OBJECT_UM2   = 50.0     # 4x is low-res; tune after viewing the sanity-check cell
MIN_PEAK_DIST    = 3        # watershed seed spacing (px); small at 4x

# FIXED DAB optical-density cutoff for "PrP-positive". A fixed value (not per-image Otsu)
# is essential so burden is COMPARABLE across images/groups. Calibrate it: a WT control
# should read ~0% burden; if controls show signal, raise it. ~0.05 fit the deer cerebellum.
DAB_THRESHOLD    = 0.05

REGIONS = ['cerebellum', 'hippocampus', 'midbrain', 'septum']
MAGS    = ['4x']
# Single 4x scale, applied to every image. Filled by notebook 00 (scale_table.csv),
# or set the real value here, e.g. {'4x': 1.72}  (um per pixel at 4x on your scope).
MAG_TO_UM_PER_PX = {'4x': np.nan}

IMG_EXTS = {'.tif', '.tiff'}
# de-duplicate by filename: controls/GtElk is a byte-identical COPY of GtElk/
_seen = set(); image_paths = []
for p in sorted(DATA_DIR.rglob('*')):
    if p.suffix.lower() in IMG_EXTS and p.name not in _seen:
        _seen.add(p.name); image_paths.append(p)
print(f'Discovered {len(image_paths)} unique images (deduplicated) under {DATA_DIR}')

In [ ]:
# --- per-image scale from notebook 00 (run 00_calibrate_scalebars.ipynb first) ---
try:
    _SCALE = pd.read_csv('outputs_calib/scale_table.csv').set_index('image')['um_per_px'].to_dict()
    print(f'Loaded {len(_SCALE)} per-image scales from outputs_calib/scale_table.csv')
except Exception:
    _SCALE = {}
    print('No scale_table.csv yet — run notebook 00 first')

# image stem -> animal id, from animal_key.csv (built from the original-filename READMEs).
# Keyed by stem because parse_metadata looks up path.stem.
try:
    _ak = pd.read_csv('animal_key.csv')
    _ANIMAL_KEY = {Path(im).stem: a for im, a in zip(_ak['image'], _ak['animal'])}
    n_an = len(set(v for v in _ANIMAL_KEY.values() if v != 'UNKNOWN'))
    print(f'Loaded {len(_ANIMAL_KEY)} image->animal mappings ({n_an} distinct animals)')
except Exception:
    _ANIMAL_KEY = {}
    print('No animal_key.csv — stats stay image-level')

def parse_metadata(path):
    name = path.stem
    low = name.lower()
    if   low.startswith('gtdeer'): species = 'deer'
    elif low.startswith('gtelk'):  species = 'elk'
    elif low.startswith('wt'):     species = 'wt'
    else:                          species = 'unknown'
    condition = 'control' if 'control' in low else ('treatment' if 'treatment' in low else 'unknown')
    region    = next((r for r in REGIONS if r in low), 'unknown')
    mag       = next((m for m in MAGS if m in low), None)
    image_id  = low.split('_')[-1]
    animal    = _ANIMAL_KEY.get(name, 'UNKNOWN')
    return dict(species=species, condition=condition, region=region,
                magnification=mag, image_id=image_id, animal=animal)

def get_um_per_px(path, meta):
    return _SCALE.get(path.name, np.nan)

def load_and_standardize(path, meta):
    img = io.imread(path)
    if img.ndim == 2:
        img = color.gray2rgb(img)
    img = img[..., :3]
    upp = get_um_per_px(path, meta)
    if TARGET_UM_PER_PX is None or np.isnan(upp):
        if np.isnan(upp):
            warnings.warn(f'No scale for {path.name} — areas will be in PIXELS')
        return img, upp
    scale = upp / TARGET_UM_PER_PX
    if abs(scale - 1) > 1e-3:
        img = rescale(img, scale, channel_axis=-1, anti_aliasing=True,
                      preserve_range=True).astype(np.uint8)
    return img, TARGET_UM_PER_PX

In [ ]:
def dab_channel(img_rgb):
    return color.rgb2hed(util.img_as_float(img_rgb))[..., 2]

def tissue_mask(img_rgb):
    # tissue = not-white background AND not-black annotations (scale bar / text)
    g = color.rgb2gray(util.img_as_float(img_rgb))
    return (g < 0.92) & (g > 0.05)

def segment_dab(img_rgb, eff_upp, dab_thresh=DAB_THRESHOLD):
    dab = dab_channel(img_rgb)
    tis = tissue_mask(img_rgb)
    blank = np.zeros(dab.shape, int)
    if tis.sum() == 0:
        return blank, blank.astype(bool), tis, 0.0, np.nan
    t = filters.threshold_otsu(dab[tis]) if dab_thresh is None else dab_thresh
    pos = (dab > t) & tis
    pos = morphology.remove_small_holes(pos, max_size=64)
    area_frac = pos.sum() / tis.sum()
    if pos.sum() == 0:
        return blank, pos, tis, area_frac, t
    dist = ndi.distance_transform_edt(pos)
    peaks = peak_local_max(dist, min_distance=MIN_PEAK_DIST, labels=pos)
    markers = np.zeros(dist.shape, int)
    for i, (r, c) in enumerate(peaks, start=1):
        markers[r, c] = i
    labels = segmentation.watershed(-dist, markers, mask=pos)
    if not np.isnan(eff_upp):
        min_px = int(max(MIN_OBJECT_UM2 / (eff_upp ** 2), 1))
        labels = morphology.remove_small_objects(labels, max_size=min_px)
    return labels, pos, tis, area_frac, t

def object_features(labels, eff_upp):
    if labels.max() == 0:
        return pd.DataFrame()
    props = measure.regionprops_table(labels, properties=[
        'label', 'area', 'perimeter', 'eccentricity', 'solidity', 'extent',
        'axis_major_length', 'axis_minor_length', 'centroid'])
    df = pd.DataFrame(props)
    upp = eff_upp if not np.isnan(eff_upp) else 1.0
    df['area_um2'] = df['area'] * upp ** 2
    df['perimeter_um'] = df['perimeter'] * upp
    df['circularity'] = 4 * np.pi * df['area'] / (df['perimeter'] ** 2 + 1e-9)
    df['aspect_ratio'] = df['axis_major_length'] / (df['axis_minor_length'] + 1e-9)
    return df

## Sanity-check on a single image
Confirm the DAB channel, positive mask, and object split look right. Tune `DAB_THRESHOLD`,
`MIN_OBJECT_UM2`, `MIN_PEAK_DIST` before batch-running.

In [ ]:
if not image_paths:
    raise SystemExit('Set DATA_DIR in the CONFIG cell first.')

p = image_paths[0]                      # or index a known-positive field
meta = parse_metadata(p)
img, upp = load_and_standardize(p, meta)
labels, pos, tis, area_frac, t = segment_dab(img, upp)
print(meta, '| um/px =', upp, '| burden% =', round(100 * area_frac, 2),
      '| n_objects =', int(labels.max()))

dab = dab_channel(img)
v1, v99 = np.percentile(dab, 1), np.percentile(dab, 99.5)   # percentile-scale so it's visible
fig, axs = plt.subplots(1, 4, figsize=(18, 5))
axs[0].imshow(img);                                        axs[0].set_title('RGB')
axs[1].imshow(dab, cmap='magma', vmin=v1, vmax=v99);       axs[1].set_title('DAB channel')
axs[2].imshow(pos, cmap='gray');                           axs[2].set_title(f'positive ({100*area_frac:.2f}%)')
axs[3].imshow(color.label2rgb(labels, bg_label=0));        axs[3].set_title(f'{int(labels.max())} objects')
for a in axs: a.axis('off')
plt.tight_layout()

## How the pipeline quantifies prions (visual walk-through)
Each image becomes a number through this fixed chain. The panels show it on one representative section; the printout shows the exact arithmetic.

1. **RGB** — anti-PrP IHC photomicrograph. Brown DAB = PrP^Sc (prion) deposits; blue = hematoxylin.
2. **DAB channel** — color deconvolution (`rgb2hed`) unmixes brown from blue; pixel value ~ amount of prion-positive chromogen.
3. **Tissue mask** — the denominator: real tissue only (excludes white glass and the black scale bar).
4. **Prion-positive mask** — pixels with DAB optical density above the fixed `DAB_THRESHOLD` (0.05). Same cutoff for every image → comparable.
5. **Counted deposits** — distance-transform + watershed split the positive mask into discrete objects → count, density, size, shape.

**Primary readout — BURDEN = prion-positive pixels ÷ tissue pixels × 100** (the % of tissue occupied by prion). Calibrated `µm/px` (Olympus SIS metadata) converts pixel areas to mm² for density. DAB is semi-quantitative, so burden is a *relative, within-study* measure — hence one staining batch, one fixed threshold, and group comparisons.

In [ ]:
# ===== PIPELINE EXPLAINER: how one image becomes a burden number =====
# Pick a clearly-positive section (deer septum, burden nearest ~3%) to illustrate each step.
_cands = [q for q in image_paths if 'gtdeer' in q.name.lower() and 'septum' in q.name.lower()]
_cands = (_cands or image_paths)[:30]
_pick, _best = None, 1e9
for q in _cands:
    _m = parse_metadata(q); _im, _u = load_and_standardize(q, _m)
    _lab, _pos, _tis, _af, _t = segment_dab(_im, _u)
    if abs(100 * _af - 3) < _best:
        _best = abs(100 * _af - 3)
        _pick = (q, _im, _u, _lab, _pos, _tis, _af, _t)
q, im, u, lab, pos, tis, af, thr = _pick
dab = dab_channel(im)
v1, v99 = np.percentile(dab, 1), np.percentile(dab, 99.5)

burden = 100 * af
n = int(lab.max())
tissue_mm2 = tis.sum() * (u ** 2) / 1e6 if not np.isnan(u) else np.nan
dens = n / tissue_mm2 if tissue_mm2 else np.nan

fig, axs = plt.subplots(1, 5, figsize=(23, 5.2))
axs[0].imshow(im);                                   axs[0].set_title('1. RGB photomicrograph\nbrown = anti-PrP DAB (= prion)')
axs[1].imshow(dab, cmap='magma', vmin=v1, vmax=v99); axs[1].set_title('2. DAB channel\ncolor-deconvolved; bright = prion')
axs[2].imshow(tis, cmap='gray');                     axs[2].set_title('3. Tissue mask\nexclude glass + scale bar')
axs[3].imshow(pos, cmap='gray');                     axs[3].set_title(f'4. Prion-positive mask\nDAB OD > {thr}')
axs[4].imshow(color.label2rgb(lab, bg_label=0));     axs[4].set_title(f'5. Counted deposits\nwatershed -> {n} objects')
for a in axs: a.axis('off')
fig.suptitle(f'{q.name}    |    {u:.2f} um/px    |    '
             f'BURDEN = {burden:.2f}% positive area    |    {n} deposits    |    {dens:.0f}/mm^2',
             fontsize=13, y=1.04)
plt.tight_layout()

print('HOW THE NUMBER IS MADE:')
print(f'  burden  = prion-positive pixels / tissue pixels x 100')
print(f'          = {int(pos.sum()):,} / {int(tis.sum()):,} x 100 = {burden:.2f}%')
print(f'  count   = number of watershed objects = {n}')
print(f'  density = count / tissue area = {n} / {tissue_mm2:.1f} mm^2 = {dens:.0f} per mm^2')

## Calibrate the DAB threshold
Samples WT controls and deer/elk treatment images, sweeps the cutoff. Pick the lowest
threshold where WT control drops to its background floor while treatment stays well above.

In [ ]:
def _sample(pred, k=5):
    g = [p for p in image_paths if pred(parse_metadata(p))]
    return g[::max(1, len(g) // k)][:k] if g else []

groups = {
    'wt_control': _sample(lambda m: m['condition'] == 'control'),
    'deer_tx':    _sample(lambda m: m['species'] == 'deer' and m['condition'] == 'treatment'),
    'elk_tx':     _sample(lambda m: m['species'] == 'elk'  and m['condition'] == 'treatment'),
}
thrs = np.round(np.arange(0.02, 0.161, 0.01), 3)
rows = []
for label, paths in groups.items():
    for p in paths:
        meta = parse_metadata(p)
        img, upp = load_and_standardize(p, meta)
        dab = dab_channel(img); tis = tissue_mask(img); denom = max(tis.sum(), 1)
        for thr in thrs:
            rows.append(dict(group=label, image=p.name, thr=thr,
                             burden=100 * (((dab > thr) & tis).sum() / denom)))
cal = pd.DataFrame(rows)
pivot = cal.groupby(['thr', 'group'])['burden'].mean().unstack('group')
display(pivot.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
for grp in pivot.columns:
    ax.plot(pivot.index, pivot[grp], marker='o', label=grp)
ax.axvline(DAB_THRESHOLD, color='k', ls='--', label=f'current = {DAB_THRESHOLD}')
ax.set_xlabel('DAB threshold (optical density)'); ax.set_ylabel('mean burden % (sampled images)')
ax.set_yscale('log'); ax.legend(); ax.set_title('WT control -> background floor; treatment stays above')
plt.tight_layout()

## Batch process the whole cohort
Writes per-image and per-object CSVs. Re-run after tuning the threshold.

In [ ]:
def process_image(p):
    meta = parse_metadata(p)
    img, upp = load_and_standardize(p, meta)
    labels, pos, tis, area_frac, t = segment_dab(img, upp)
    odf = object_features(labels, upp)
    if len(odf):
        odf.insert(0, 'image', p.name)
        for k, v in meta.items():
            odf[k] = v
    if not np.isnan(upp):
        tissue_mm2 = tis.sum() * (upp ** 2) / 1e6
        density = len(odf) / tissue_mm2 if tissue_mm2 > 0 else np.nan
    else:
        tissue_mm2, density = np.nan, np.nan
    summary = dict(image=p.name, **meta, um_per_px=upp,
                   pct_area_burden=100 * area_frac,
                   n_objects=len(odf), tissue_mm2=tissue_mm2,
                   density_per_mm2=density,
                   mean_area_um2=odf['area_um2'].mean() if len(odf) else np.nan,
                   median_circularity=odf['circularity'].median() if len(odf) else np.nan,
                   dab_threshold=t)
    return summary, odf

if not image_paths:
    raise SystemExit('image_paths is empty — run the CONFIG cell first.')
print(f'Processing {len(image_paths)} images at DAB_THRESHOLD={DAB_THRESHOLD} (~2-3 min)...', flush=True)

rows, objs = [], []
for i, p in enumerate(image_paths):
    try:
        s, odf = process_image(p)
        rows.append(s)
        if len(odf): objs.append(odf)
    except Exception as e:
        print(f'[skip] {p.name}: {e}', flush=True)
    if (i + 1) % 25 == 0:
        print(f'  processed {i + 1}/{len(image_paths)}', flush=True)

img_df = pd.DataFrame(rows)
objs_df = pd.concat(objs, ignore_index=True) if objs else pd.DataFrame()
img_df.to_csv(OUT_DIR / 'per_image_summary.csv', index=False)
objs_df.to_csv(OUT_DIR / 'per_object_features.csv', index=False)
print(f'DONE. {len(img_df)} images, {len(objs_df)} objects. Saved to {OUT_DIR.resolve()}', flush=True)
img_df.head()

In [ ]:
# Group-level summary + jittered strip plot (image-level; the animal is the unit for stats).
if len(img_df):
    summary = (img_df.groupby(['species', 'condition', 'region'])
               .agg(n_images=('image', 'size'),
                    mean_pct_burden=('pct_area_burden', 'mean'),
                    mean_density=('density_per_mm2', 'mean'))
               .reset_index())
    display(summary)

    img_df['group'] = img_df['species'] + '_' + img_df['condition']
    regs = [r for r in REGIONS if r in img_df['region'].unique()]
    grps = sorted(img_df['group'].unique())
    offsets = np.linspace(-0.28, 0.28, len(grps)) if len(grps) > 1 else np.array([0.0])
    rng = np.random.default_rng(0)

    fig, ax = plt.subplots(figsize=(10, 5))
    for gi, grp in enumerate(grps):
        for ri, reg in enumerate(regs):
            y = img_df[(img_df['group'] == grp) & (img_df['region'] == reg)]['pct_area_burden'].dropna().values
            x = ri + offsets[gi] + rng.uniform(-0.05, 0.05, size=len(y))   # dodge by group + jitter
            ax.scatter(x, y, s=14, alpha=0.45, color=f'C{gi}', label=grp if ri == 0 else None)
        means = [img_df[(img_df['group'] == grp) & (img_df['region'] == r)]['pct_area_burden'].mean() for r in regs]
        ax.scatter(np.arange(len(regs)) + offsets[gi], means, marker='_', s=420, color=f'C{gi}', linewidths=2.5)
    ax.set_xticks(range(len(regs))); ax.set_xticklabels(regs)
    ax.set_xlabel('region'); ax.set_ylabel('PrP-positive area (%)')
    ax.set_title('CWD PrP burden by region / group (jittered; long bars = group mean)')
    ax.legend(title='group'); plt.tight_layout()

## Morphotype typing (plaque type)
Cluster objects by shape into compact / intermediate / diffuse, then count per group & region.

In [ ]:
# pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

FEAT = ['area_um2', 'circularity', 'solidity', 'aspect_ratio', 'eccentricity', 'extent']
N_MORPHOTYPES = 3

md = objs_df.dropna(subset=FEAT).copy()
X = StandardScaler().fit_transform(md[FEAT].values)
Z = PCA(n_components=2).fit_transform(X)
md['cluster'] = KMeans(n_clusters=N_MORPHOTYPES, n_init=10, random_state=0).fit_predict(X)
order = md.groupby('cluster')['area_um2'].median().sort_values().index
names = {lab: nm for lab, nm in zip(order, ['compact', 'intermediate', 'diffuse'][:N_MORPHOTYPES])}
md['morphotype'] = md['cluster'].map(names)
objs_df = objs_df.drop(columns=['morphotype'], errors='ignore').merge(
    md[['image', 'label', 'morphotype']], on=['image', 'label'], how='left')
objs_df.to_csv(OUT_DIR / 'per_object_features.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for nm in names.values():
    m = md['morphotype'].values == nm
    ax[0].scatter(Z[m, 0], Z[m, 1], s=6, alpha=0.4, label=nm)
ax[0].set_xlabel('PC1'); ax[0].set_ylabel('PC2'); ax[0].legend(); ax[0].set_title('morphotype clusters (PCA)')
md.boxplot(column='area_um2', by='morphotype', ax=ax[1]); ax[1].set_yscale('log')
ax[1].set_title('area by morphotype'); plt.suptitle(''); plt.tight_layout()
print(md.groupby('morphotype')[FEAT].median())

In [ ]:
# Morphotype composition per group/region (counts) + jittered per-image fractions.
comp = (objs_df.dropna(subset=['morphotype'])
        .pivot_table(index=['species', 'condition', 'region'],
                     columns='morphotype', values='label', aggfunc='count', fill_value=0))
display(comp)

mt = objs_df.dropna(subset=['morphotype']).copy()
per_img = (mt.groupby(['image', 'species', 'condition', 'morphotype']).size().rename('n').reset_index())
wide = per_img.pivot_table(index=['image', 'species', 'condition'],
                           columns='morphotype', values='n', fill_value=0)
wide = wide.div(wide.sum(axis=1), axis=0).reset_index()          # row-normalize -> fractions
wide['group'] = wide['species'] + '_' + wide['condition']

morphs = [m for m in ['compact', 'intermediate', 'diffuse'] if m in wide.columns]
grps = sorted(wide['group'].unique())
offs = np.linspace(-0.28, 0.28, len(grps)) if len(grps) > 1 else np.array([0.0])
rng = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=(10, 5))
for gi, grp in enumerate(grps):
    sub = wide[wide['group'] == grp]
    for mi, mph in enumerate(morphs):
        y = sub[mph].values
        x = mi + offs[gi] + rng.uniform(-0.05, 0.05, size=len(y))
        ax.scatter(x, y, s=12, alpha=0.4, color=f'C{gi}', label=grp if mi == 0 else None)
    means = [sub[mph].mean() for mph in morphs]
    ax.scatter(np.arange(len(morphs)) + offs[gi], means, marker='_', s=420, color=f'C{gi}', linewidths=2.5)
ax.set_xticks(range(len(morphs))); ax.set_xticklabels(morphs)
ax.set_xlabel('morphotype'); ax.set_ylabel('per-image fraction of objects')
ax.set_title('Morphotype composition per image, by group (jittered; long bars = group mean)')
ax.legend(title='group'); plt.tight_layout()

## Spread — Ripley's L(r)
Per-image clustering of deposit centroids vs complete spatial randomness (CSR). `L(r)-r`
above the envelope = clustering. Self-contained (no edge correction — use R `spatstat`
`Kinhom`/`pcf` for publication-grade inhomogeneous stats).

In [ ]:
from scipy.spatial import cKDTree

def ripley_L(xy, width, height, radii, n_sim=99, seed=0):
    rng = np.random.default_rng(seed)
    n = len(xy); area = width * height
    if n < 5 or area <= 0:
        return None
    lam = n / area
    def L_of(pts):
        tree = cKDTree(pts)
        K = np.array([2 * len(tree.query_pairs(r, output_type='ndarray')) / (lam * n) for r in radii])
        return np.sqrt(K / np.pi) - radii
    L_obs = L_of(xy)
    sims = np.array([L_of(np.column_stack([rng.uniform(0, width, n), rng.uniform(0, height, n)]))
                     for _ in range(n_sim)])
    lo, hi = np.percentile(sims, [2.5, 97.5], axis=0)
    return L_obs, lo, hi

cand = img_df.sort_values('n_objects', ascending=False)['image'].iloc[0]
pp = next(p for p in image_paths if p.name == cand)
meta = parse_metadata(pp); img, upp = load_and_standardize(pp, meta)
labels, pos, tis, af, t = segment_dab(img, upp)
u = upp if not np.isnan(upp) else 1.0
props = measure.regionprops_table(labels, properties=['centroid'])
xy = np.column_stack([props['centroid-1'], props['centroid-0']]) * u
H, W = labels.shape[0] * u, labels.shape[1] * u
radii = np.linspace(5, min(H, W) / 4, 25)
res = ripley_L(xy, W, H, radii)
if res is None:
    print('Too few objects for spatial analysis on', cand)
else:
    L, lo, hi = res
    plt.figure(figsize=(7, 5))
    plt.fill_between(radii, lo, hi, color='gray', alpha=0.3, label='CSR 95% envelope')
    plt.plot(radii, L, 'b-', label='observed')
    plt.axhline(0, color='k', lw=0.8)
    plt.xlabel('r (um)'); plt.ylabel('L(r) - r'); plt.legend()
    plt.title(f'Ripley L — {cand} (n={len(xy)}); above envelope = clustering')

## Group comparison
With animal IDs (`_ANIMAL_KEY` populated) this fits a mixed-effects model (animal = random
effect) — the publication-valid analysis. Until then it falls back to an **image-level**
comparison (Kruskal-Wallis per region) which is EXPLORATORY and pseudoreplicated.

In [ ]:
# pip install statsmodels  --  ANIMAL-LEVEL stats (animal = experimental unit)
from scipy import stats
import statsmodels.formula.api as smf

known = img_df[img_df['animal'] != 'UNKNOWN'].copy()
# roll up images -> one value per (animal, species, region)
agg = (known.groupby(['animal', 'species', 'region'], as_index=False)
       .agg(burden=('pct_area_burden', 'mean')))
print('animals per species:', agg.groupby('species')['animal'].nunique().to_dict())

if agg['animal'].nunique() >= 3:
    # (1) mixed-effects model: region-adjusted species effect, animal as random intercept
    model = smf.mixedlm('burden ~ C(species) + C(region)', agg, groups=agg['animal'])
    print(model.fit().summary())

    # (2) key pairwise test: deer vs elk per region, on ANIMAL means (Mann-Whitney)
    print('\ndeer vs elk (animal-level Mann-Whitney, per region):')
    for reg in [r for r in REGIONS if r in agg['region'].unique()]:
        d = agg[(agg['species'] == 'deer') & (agg['region'] == reg)]['burden']
        e = agg[(agg['species'] == 'elk')  & (agg['region'] == reg)]['burden']
        if len(d) > 1 and len(e) > 1:
            U, p = stats.mannwhitneyu(d, e, alternative='two-sided')
            print(f'  {reg:12s} deer n={len(d)} (med {d.median():.2f}%)  '
                  f'elk n={len(e)} (med {e.median():.2f}%)  p={p:.3g}')

    # (3) ANIMAL-level jittered figure (each point = one mouse)
    regs = [r for r in REGIONS if r in agg['region'].unique()]
    grps = sorted(agg['species'].unique())
    offs = np.linspace(-0.28, 0.28, len(grps)) if len(grps) > 1 else np.array([0.0])
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(10, 5))
    for gi, grp in enumerate(grps):
        for ri, reg in enumerate(regs):
            y = agg[(agg['species'] == grp) & (agg['region'] == reg)]['burden'].values
            x = ri + offs[gi] + rng.uniform(-0.05, 0.05, size=len(y))
            ax.scatter(x, y, s=24, alpha=0.6, color=f'C{gi}', label=grp if ri == 0 else None)
        means = [agg[(agg['species'] == grp) & (agg['region'] == r)]['burden'].mean() for r in regs]
        ax.scatter(np.arange(len(regs)) + offs[gi], means, marker='_', s=460, color=f'C{gi}', linewidths=3)
    ax.set_xticks(range(len(regs))); ax.set_xticklabels(regs)
    ax.set_xlabel('region'); ax.set_ylabel('PrP burden (%) — per-animal mean')
    ax.set_title('Animal-level CWD PrP burden by region / group (each point = 1 mouse)')
    ax.legend(title='species'); plt.tight_layout()
else:
    print('Too few mapped animals — check animal_key.csv loaded in the helpers cell.')

## What's left / Phase 2

- **Phase 2 DL** (only where classical fails on confluent diffuse staining): annotate a few
  dozen fields with **micro_sam** (napari), then train **StarDist** or **nnU-Net** — not Cellpose.
  Split **by animal**.
- **Atlas registration** for whole-section images (DeepSlice -> QuickNII -> VisuAlign, or ABBA).
- **Publication-grade spatial stats**: R `spatstat` `Kinhom`/`pcf` with edge correction.